In [10]:
import joblib
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from roadmap import PROFESSION_CATEGORIES
from roadmap import build_profession_profile
import pandas as pd
from collections import defaultdict, Counter
from skills_taxonomy import STOP_WORDS, normalize_skill
import ast

In [2]:
# Load the normalized job data
df = pd.read_csv('../data/raw/data_jobs_norm.csv')

In [3]:
# Remove rows with missing 'job_skills'
df.dropna(subset=['job_skills'], inplace=True)

In [4]:
# Fucntions for vectorizer notebook
def get_top_skills_by_category(
    df, 
    profession: str, 
    top_n: int = 5,
    stop_words = STOP_WORDS
) -> dict:
    counter_by_cat = defaultdict(Counter)
    
    for skills_str in df[df['job_title_unified'] == profession]['job_type_skills']:
        parsed = ast.literal_eval(skills_str)
        for category, skills in parsed.items():
            filtered = [s for s in skills if s.lower() not in stop_words]
            counter_by_cat[category].update(filtered)
    
    result = {}
    for cat, counter in counter_by_cat.items():
        selected = []
        seen_groups = set()  
        
        for skill, _ in counter.most_common(top_n * 3): 
            if len(selected) >= top_n:
                break
            selected.append(skill)
        
        result[cat] = selected
    
    return result

In [5]:
def build_profession_string(
    skills_by_category: dict,
    profession: str,
) -> str:
    """
    Builds a string for TF-IDF considering:
    1. Only relevant categories for the profession
    2. Order of categories in PROFESSION_CATEGORIES — first ones are more important
    """
    relevant_cats = PROFESSION_CATEGORIES.get(profession, list(skills_by_category.keys()))
    
    tokens = []
    n_cats = len(relevant_cats)
    
    for rank, cat in enumerate(relevant_cats):
        if cat not in skills_by_category:
            continue
        
        base_weight = 1
        
        positional_weight = 1.0 - (rank / n_cats) * 0.5

        final_weight = max(1, round(base_weight * positional_weight))
        
        skills = skills_by_category[cat]
        normalized = [s.replace(' ', '_').lower() for s in skills]
        tokens.extend(normalized * final_weight)
    
    return ' '.join(tokens)

def train_and_save_tfidf(
    df,
    output_dir: str = 'models/',
):
    """
    Train a TF-IDF vectorizer on profession profiles and save the vectorizer, profession vectors, and names.
    """
    from pathlib import Path
    Path(output_dir).mkdir(exist_ok=True)
    
    professions = df['job_title_unified'].unique().tolist()

    profession_strings = {}
    for profession in professions:
        skills_by_cat = get_top_skills_by_category(df, profession, top_n=3)
        profile_string = build_profession_string(skills_by_cat, profession)
        profession_strings[profession] = profile_string
        print(f"\n{profession}:\n  {profile_string}")

    corpus = list(profession_strings.values())
    profession_names = list(profession_strings.keys())
    
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),  
        min_df=1,
        sublinear_tf=True,  
    )
    profession_vectors = vectorizer.fit_transform(corpus)
    
    # Сохраняем
    with open(f'{output_dir}/tfidf_vectorizer.pkl', 'wb') as f:
        pickle.dump(vectorizer, f)
    
    with open(f'{output_dir}/profession_vectors.pkl', 'wb') as f:
        pickle.dump(profession_vectors, f)
    
    with open(f'{output_dir}/profession_names.pkl', 'wb') as f:
        pickle.dump(profession_names, f)
    
    with open(f'{output_dir}/profession_strings.json', 'w', encoding='utf-8') as f:
        json.dump(profession_strings, f, ensure_ascii=False, indent=2)
    
    print(f"\n✓ Vectorizer trained on {len(professions)} professions")
    print(f"✓ Dictionary size: {len(vectorizer.vocabulary_)} tokens")
    
    return vectorizer, profession_vectors, profession_names

In [6]:
def get_skill_scores(
    student_raw_skills: list,
    vectorizer,
    profession_vectors,
    profession_names: list,
) -> dict:
    """
    Calculates cosine similarity between student skills and profession profiles.
    """

    student_skills = set()
    for raw in student_raw_skills:
        normalized = normalize_skill(raw)
        if normalized and normalized not in STOP_WORDS:
            student_skills.add(normalized)
            
    if not student_skills:
        return {name: 0.0 for name in profession_names}

    student_doc = " ".join(student_skills)

    student_vector = vectorizer.transform([student_doc]) 

    scores = cosine_similarity(student_vector, profession_vectors)[0]

    return dict(sorted(
        zip(profession_names, scores.tolist()),
        key=lambda x: x[1],
        reverse=True
    ))

In [7]:
vectorizer, profession_vectors, profession_names = train_and_save_tfidf(df)
# Test
test_skills = ["python", "sql", "tensorflow", "pytorch", "aws", "spark"]
scores = get_skill_scores(
    test_skills, vectorizer, profession_vectors, profession_names
)
print("\nScores:")
for prof, score in scores.items():
    print(f"  {prof}: {score:.3f}")


Data Analyst:
  sql python r excel tableau power_bi sql_server mysql postgresql

Data Engineer:
  sql python java sql_server mysql postgresql aws azure snowflake git docker kubernetes spark kafka hadoop

Business Analyst:
  sql python r excel tableau power_bi sql_server mysql postgresql

Data Scientist:
  python sql r spark tensorflow pytorch sql_server mysql mongodb aws azure gcp git docker kubernetes

Machine Learning Engineer:
  python sql java pytorch tensorflow spark aws azure gcp docker kubernetes git

Cloud Engineer:
  python sql java aws azure gcp terraform kubernetes docker sql_server mysql postgresql

Software Engineer:
  python sql java angular ruby node.js mysql postgresql sql_server aws azure gcp kubernetes docker git

✓ Vectorizer trained on 7 professions
✓ Dictionary size: 75 tokens

Scores:
  Data Scientist: 0.482
  Machine Learning Engineer: 0.379
  Data Engineer: 0.148
  Cloud Engineer: 0.104
  Software Engineer: 0.079
  Data Analyst: 0.071
  Business Analyst: 0.071


In [9]:
test_da = ["python","sql"]
scores = get_skill_scores(test_da, vectorizer, profession_vectors, profession_names)
for prof, score in scores.items():
    print(f"  {prof}: {score:.3f}")

  Data Analyst: 0.349
  Business Analyst: 0.349
  Data Engineer: 0.224
  Cloud Engineer: 0.115
  Machine Learning Engineer: 0.107
  Data Scientist: 0.098
  Software Engineer: 0.087


In [12]:
joblib.dump(vectorizer, '../models/tfidf_vectorizer.joblib')
joblib.dump(profession_vectors, '../models/profession_vectors.joblib')
joblib.dump(profession_names, '../models/profession_names.joblib')

['../models/profession_names.joblib']